# 🎯 Feature Extraction and Representation Workshop

**CMSC 178IP - Digital Image Processing**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/CMSC178IP/blob/main/07%20-%20Feature%20Extraction/notebooks/feature_extraction_workshop.ipynb)

---

## 📋 Learning Objectives

By the end of this workshop, you will be able to:

1. **Implement edge detection algorithms** from scratch and using built-in functions
2. **Apply various gradient operators** (Sobel, Prewitt, Roberts) to detect edges
3. **Use the Hough transform** for line and circle detection
4. **Extract texture features** using Local Binary Patterns and Gabor filters
5. **Detect corners and keypoints** using Harris, SIFT, and ORB methods
6. **Match features** between images for recognition and registration
7. **Analyze real-world applications** of feature extraction techniques

---

## ⚙️ Setup and Imports

In [ ]:
# Install required packages if running on Colab
import sys
if 'google.colab' in sys.modules:
    !pip install opencv-python-headless scikit-image matplotlib seaborn

# Core imports
import numpy as np
import matplotlib.pyplot as plt
import cv2
from scipy import ndimage
from skimage import feature, filters, morphology, transform, data, exposure
from skimage.data import camera, coins, checkerboard, coffee
import seaborn as sns
from IPython.display import display, Image

# Set up plotting
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("✅ All packages imported successfully!")
print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")

---

## 📊 Part 1: Understanding Edge Detection

Edge detection is fundamental to feature extraction. Let's start by understanding how edges are formed and detected in digital images.

In [ ]:
# Load a test image
image = camera()  # Classic test image

# Display the original image
plt.figure(figsize=(10, 6))
plt.subplot(1, 2, 1)
plt.imshow(image, cmap='gray')
plt.title('Original Camera Image')
plt.axis('off')

# Show histogram to understand intensity distribution
plt.subplot(1, 2, 2)
plt.hist(image.ravel(), bins=256, alpha=0.7, color='blue')
plt.title('Intensity Histogram')
plt.xlabel('Pixel Intensity')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Image shape: {image.shape}")
print(f"Data type: {image.dtype}")
print(f"Intensity range: [{image.min()}, {image.max()}]")

### 🔍 Understanding Image Gradients

An edge occurs where there's a significant change in image intensity. We detect these changes using gradients:

- **Gradient magnitude**: $|\nabla I| = \sqrt{(\frac{\partial I}{\partial x})^2 + (\frac{\partial I}{\partial y})^2}$
- **Gradient direction**: $\theta = \arctan(\frac{\partial I/\partial y}{\partial I/\partial x})$

In [ ]:
def apply_sobel_operator(image):
    """Apply Sobel operator to detect edges."""
    # Define Sobel kernels
    sobel_x = np.array([[-1, 0, 1], 
                        [-2, 0, 2], 
                        [-1, 0, 1]])
    
    sobel_y = np.array([[-1, -2, -1], 
                        [0,  0,  0], 
                        [1,  2,  1]])
    
    # Apply convolution
    grad_x = ndimage.convolve(image.astype(float), sobel_x)
    grad_y = ndimage.convolve(image.astype(float), sobel_y)
    
    # Calculate magnitude and direction
    magnitude = np.sqrt(grad_x**2 + grad_y**2)
    direction = np.arctan2(grad_y, grad_x)
    
    return grad_x, grad_y, magnitude, direction

# Apply Sobel operator
grad_x, grad_y, magnitude, direction = apply_sobel_operator(image)

# Visualize results
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(image, cmap='gray')
axes[0, 0].set_title('Original Image')
axes[0, 0].axis('off')

axes[0, 1].imshow(grad_x, cmap='gray')
axes[0, 1].set_title('Gradient X (Vertical Edges)')
axes[0, 1].axis('off')

axes[0, 2].imshow(grad_y, cmap='gray')
axes[0, 2].set_title('Gradient Y (Horizontal Edges)')
axes[0, 2].axis('off')

axes[1, 0].imshow(magnitude, cmap='gray')
axes[1, 0].set_title('Gradient Magnitude')
axes[1, 0].axis('off')

axes[1, 1].imshow(direction, cmap='hsv')
axes[1, 1].set_title('Gradient Direction')
axes[1, 1].axis('off')

# Show the kernels
kernel_display = np.hstack([sobel_x, np.zeros((3, 1)), sobel_y])
im = axes[1, 2].imshow(kernel_display, cmap='RdBu', vmin=-2, vmax=2)
axes[1, 2].set_title('Sobel Kernels (X and Y)')
axes[1, 2].axis('off')

# Add kernel values as text
for i in range(3):
    for j in range(3):
        axes[1, 2].text(j, i, f'{sobel_x[i, j]}', ha='center', va='center', 
                       color='white' if abs(sobel_x[i, j]) > 1 else 'black', fontweight='bold')
        axes[1, 2].text(j+4, i, f'{sobel_y[i, j]}', ha='center', va='center', 
                       color='white' if abs(sobel_y[i, j]) > 1 else 'black', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Gradient magnitude range: [{magnitude.min():.2f}, {magnitude.max():.2f}]")
print(f"Strong edges (magnitude > {magnitude.mean() + magnitude.std():.2f}): {np.sum(magnitude > magnitude.mean() + magnitude.std())} pixels")

### 🔬 Comparing Different Edge Operators

Let's compare the performance of different gradient operators on the same image.

In [ ]:
def compare_edge_operators(image):
    """Compare different edge detection operators."""
    
    # Define operators
    operators = {
        'Sobel': {'x': np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]),
                  'y': np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]])},
        
        'Prewitt': {'x': np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]]),
                    'y': np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]])},
        
        'Roberts': {'x': np.array([[1, 0], [0, -1]]),
                    'y': np.array([[0, 1], [-1, 0]])}
    }
    
    results = {}
    
    for name, kernels in operators.items():
        grad_x = ndimage.convolve(image.astype(float), kernels['x'])
        grad_y = ndimage.convolve(image.astype(float), kernels['y'])
        magnitude = np.sqrt(grad_x**2 + grad_y**2)
        results[name] = magnitude
    
    return results

# Compare operators
edge_results = compare_edge_operators(image)

# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(image, cmap='gray')
axes[0, 0].set_title('Original Image')
axes[0, 0].axis('off')

for i, (name, result) in enumerate(edge_results.items()):
    row = (i + 1) // 2
    col = (i + 1) % 2
    axes[row, col].imshow(result, cmap='gray')
    axes[row, col].set_title(f'{name} Edge Detection')
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

# Quantitative comparison
print("Edge Detection Operator Comparison:")
print("-" * 50)
for name, result in edge_results.items():
    edge_pixels = np.sum(result > result.mean() + result.std())
    total_pixels = result.size
    edge_percentage = (edge_pixels / total_pixels) * 100
    print(f"{name:8s}: {edge_pixels:6d} edge pixels ({edge_percentage:.2f}%)")

---

## 🎯 Part 2: Advanced Edge Detection - Canny Algorithm

The Canny edge detector is considered the optimal edge detector. Let's implement and understand its multi-stage process.

In [ ]:
def canny_edge_detection_steps(image, sigma=1.0, low_threshold=0.1, high_threshold=0.2):
    """Demonstrate the steps of Canny edge detection."""
    
    # Step 1: Gaussian smoothing
    smoothed = filters.gaussian(image, sigma=sigma)
    
    # Step 2: Gradient calculation
    grad_x = filters.sobel_h(smoothed)
    grad_y = filters.sobel_v(smoothed)
    magnitude = np.sqrt(grad_x**2 + grad_y**2)
    direction = np.arctan2(grad_y, grad_x)
    
    # Step 3: Non-maximum suppression (simplified)
    suppressed = magnitude.copy()
    # This is a simplified version - actual implementation is more complex
    
    # Step 4: Double thresholding
    high_thresh = high_threshold * magnitude.max()
    low_thresh = low_threshold * magnitude.max()
    
    strong_edges = magnitude > high_thresh
    weak_edges = (magnitude >= low_thresh) & (magnitude <= high_thresh)
    
    # Step 5: Edge tracking by hysteresis (using scikit-image implementation)
    final_edges = feature.canny(image, sigma=sigma, low_threshold=low_threshold, high_threshold=high_threshold)
    
    return {
        'original': image,
        'smoothed': smoothed,
        'magnitude': magnitude,
        'direction': direction,
        'strong_edges': strong_edges,
        'weak_edges': weak_edges,
        'final_edges': final_edges
    }

# Apply Canny edge detection with steps
canny_steps = canny_edge_detection_steps(image)

# Visualize all steps
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Plot each step
step_names = ['original', 'smoothed', 'magnitude', 'direction', 
              'strong_edges', 'weak_edges', 'final_edges']
step_titles = ['Original', 'Gaussian Smoothed', 'Gradient Magnitude', 'Gradient Direction',
               'Strong Edges', 'Weak Edges', 'Final Canny Edges']

for i, (name, title) in enumerate(zip(step_names, step_titles)):
    if i < 7:  # We have 7 steps to show
        row = i // 4
        col = i % 4
        
        if name == 'direction':
            axes[row, col].imshow(canny_steps[name], cmap='hsv')
        else:
            axes[row, col].imshow(canny_steps[name], cmap='gray')
        
        axes[row, col].set_title(title)
        axes[row, col].axis('off')

# Remove the last empty subplot
axes[1, 3].remove()

plt.tight_layout()
plt.show()

# Parameter sensitivity analysis
print("Canny Edge Detection Parameter Analysis:")
print("-" * 50)
print(f"Gaussian sigma: {1.0}")
print(f"Low threshold: {0.1}")
print(f"High threshold: {0.2}")
print(f"Final edge pixels: {np.sum(canny_steps['final_edges'])}")
print(f"Edge density: {np.sum(canny_steps['final_edges']) / canny_steps['final_edges'].size * 100:.2f}%")

### 🎛️ Interactive Parameter Tuning

Let's explore how different parameters affect the Canny edge detection results.

In [ ]:
def explore_canny_parameters(image):
    """Explore the effect of different Canny parameters."""
    
    # Different parameter combinations
    params = [
        {'sigma': 0.5, 'low': 0.05, 'high': 0.15, 'title': 'Low Sigma, Low Thresholds'},
        {'sigma': 1.0, 'low': 0.1, 'high': 0.2, 'title': 'Medium Sigma, Medium Thresholds'},
        {'sigma': 2.0, 'low': 0.2, 'high': 0.4, 'title': 'High Sigma, High Thresholds'},
        {'sigma': 1.0, 'low': 0.01, 'high': 0.05, 'title': 'Medium Sigma, Very Low Thresholds'}
    ]
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    for i, param in enumerate(params):
        row = i // 2
        col = i % 2
        
        edges = feature.canny(image, sigma=param['sigma'], 
                             low_threshold=param['low'], 
                             high_threshold=param['high'])
        
        axes[row, col].imshow(edges, cmap='gray')
        axes[row, col].set_title(param['title'])
        axes[row, col].axis('off')
        
        # Add parameter info
        info_text = f"σ={param['sigma']}, L={param['low']}, H={param['high']}"
        axes[row, col].text(10, 30, info_text, color='white', fontsize=10, 
                           bbox=dict(boxstyle="round,pad=0.3", facecolor='black', alpha=0.7))
    
    plt.tight_layout()
    plt.show()
    
    # Quantitative analysis
    print("Parameter Effect Analysis:")
    print("-" * 60)
    print(f"{'Parameters':<30} {'Edge Pixels':<12} {'Percentage':<12}")
    print("-" * 60)
    
    for param in params:
        edges = feature.canny(image, sigma=param['sigma'], 
                             low_threshold=param['low'], 
                             high_threshold=param['high'])
        edge_count = np.sum(edges)
        percentage = (edge_count / edges.size) * 100
        param_str = f"σ={param['sigma']}, L={param['low']}, H={param['high']}"
        print(f"{param_str:<30} {edge_count:<12} {percentage:<12.2f}%")

explore_canny_parameters(image)

---

## 🔍 Part 3: Hough Transform for Geometric Feature Detection

The Hough transform is powerful for detecting parametric shapes like lines and circles in images.

In [ ]:
def create_test_image_with_shapes():
    """Create a test image with various geometric shapes."""
    # Create a blank image
    test_image = np.zeros((300, 400), dtype=np.uint8)
    
    # Add lines
    cv2.line(test_image, (50, 50), (350, 100), 255, 3)
    cv2.line(test_image, (100, 200), (300, 250), 255, 3)
    cv2.line(test_image, (150, 50), (200, 280), 255, 3)
    
    # Add circles
    cv2.circle(test_image, (80, 150), 30, 255, 3)
    cv2.circle(test_image, (320, 180), 25, 255, 3)
    cv2.circle(test_image, (200, 120), 20, 255, 3)
    
    # Add rectangles (for variety)
    cv2.rectangle(test_image, (250, 50), (300, 100), 255, 3)
    
    # Add some noise
    noise = np.random.normal(0, 15, test_image.shape).astype(np.uint8)
    test_image = np.clip(test_image.astype(int) + noise, 0, 255).astype(np.uint8)
    
    return test_image

# Create test image
shapes_image = create_test_image_with_shapes()

# Display the test image
plt.figure(figsize=(10, 6))
plt.imshow(shapes_image, cmap='gray')
plt.title('Test Image with Geometric Shapes')
plt.axis('off')
plt.show()

print(f"Created test image with shape: {shapes_image.shape}")
print("Contains: lines, circles, rectangles, and noise")

In [ ]:
def detect_lines_hough(image, visualize=True):
    """Detect lines using Hough transform."""
    
    # Apply edge detection first
    edges = feature.canny(image, sigma=1, low_threshold=50, high_threshold=150)
    
    # Convert to uint8 for OpenCV
    edges_uint8 = (edges * 255).astype(np.uint8)
    
    # Apply Hough line transform
    lines = cv2.HoughLines(edges_uint8, rho=1, theta=np.pi/180, threshold=80)
    
    # Apply probabilistic Hough line transform
    lines_p = cv2.HoughLinesP(edges_uint8, rho=1, theta=np.pi/180, 
                             threshold=50, minLineLength=30, maxLineGap=10)
    
    if visualize:
        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        
        # Original image
        axes[0].imshow(image, cmap='gray')
        axes[0].set_title('Original Image')
        axes[0].axis('off')
        
        # Edge detection
        axes[1].imshow(edges, cmap='gray')
        axes[1].set_title('Edge Detection')
        axes[1].axis('off')
        
        # Standard Hough lines
        hough_image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        if lines is not None:
            for line in lines:
                rho, theta = line[0]
                a = np.cos(theta)
                b = np.sin(theta)
                x0 = a * rho
                y0 = b * rho
                x1 = int(x0 + 1000 * (-b))
                y1 = int(y0 + 1000 * (a))
                x2 = int(x0 - 1000 * (-b))
                y2 = int(y0 - 1000 * (a))
                cv2.line(hough_image, (x1, y1), (x2, y2), (255, 0, 0), 2)
        
        axes[2].imshow(hough_image)
        axes[2].set_title(f'Hough Lines ({len(lines) if lines is not None else 0} detected)')
        axes[2].axis('off')
        
        # Probabilistic Hough lines
        hough_p_image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        if lines_p is not None:
            for line in lines_p:
                x1, y1, x2, y2 = line[0]
                cv2.line(hough_p_image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        axes[3].imshow(hough_p_image)
        axes[3].set_title(f'Probabilistic Hough ({len(lines_p) if lines_p is not None else 0} detected)')
        axes[3].axis('off')
        
        plt.tight_layout()
        plt.show()
    
    return lines, lines_p

# Detect lines
detected_lines, detected_lines_p = detect_lines_hough(shapes_image)

print(f"Standard Hough Transform detected: {len(detected_lines) if detected_lines is not None else 0} lines")
print(f"Probabilistic Hough Transform detected: {len(detected_lines_p) if detected_lines_p is not None else 0} line segments")

In [ ]:
def detect_circles_hough(image, visualize=True):
    """Detect circles using Hough transform."""
    
    # Apply Hough circle transform
    circles = cv2.HoughCircles(image, cv2.HOUGH_GRADIENT, dp=1, minDist=30,
                              param1=50, param2=30, minRadius=10, maxRadius=50)
    
    if visualize:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        # Original image
        axes[0].imshow(image, cmap='gray')
        axes[0].set_title('Original Image')
        axes[0].axis('off')
        
        # Detected circles
        circle_image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        
        if circles is not None:
            circles = np.round(circles[0, :]).astype("int")
            
            for (x, y, r) in circles:
                # Draw the circle
                cv2.circle(circle_image, (x, y), r, (0, 255, 0), 3)
                # Draw the center
                cv2.circle(circle_image, (x, y), 2, (255, 0, 0), 3)
        
        axes[1].imshow(circle_image)
        axes[1].set_title(f'Detected Circles ({len(circles) if circles is not None else 0} found)')
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()
    
    return circles

# Detect circles
detected_circles = detect_circles_hough(shapes_image)

if detected_circles is not None:
    print(f"Detected {len(detected_circles)} circles:")
    for i, (x, y, r) in enumerate(detected_circles):
        print(f"  Circle {i+1}: Center=({x}, {y}), Radius={r}")
else:
    print("No circles detected")

---

## 🧩 Part 4: Texture Analysis with Local Binary Patterns

Texture is an important visual cue. Let's explore Local Binary Patterns (LBP) for texture analysis.

In [ ]:
def analyze_texture_with_lbp(image, P=8, R=1):
    """Analyze texture using Local Binary Patterns."""
    
    # Apply LBP
    lbp = feature.local_binary_pattern(image, P=P, R=R, method='uniform')
    
    # Calculate LBP histogram
    hist, _ = np.histogram(lbp.ravel(), bins=P+2, range=(0, P+2))
    hist = hist.astype(float)
    hist /= (hist.sum() + 1e-7)  # Normalize
    
    return lbp, hist

def create_texture_samples():
    """Create different texture patterns for analysis."""
    size = 128
    
    # Checkerboard pattern
    checker = np.kron([[1, 0] * 8, [0, 1] * 8] * 8, np.ones((8, 8)))
    
    # Random texture
    random_texture = np.random.rand(size, size)
    
    # Sinusoidal pattern
    x = np.linspace(0, 4*np.pi, size)
    y = np.linspace(0, 4*np.pi, size)
    X, Y = np.meshgrid(x, y)
    sine_texture = np.sin(X) * np.cos(Y)
    
    # Gradient pattern
    gradient = np.linspace(0, 1, size)
    gradient_texture = np.tile(gradient, (size, 1))
    
    return {
        'Checkerboard': checker,
        'Random': random_texture,
        'Sinusoidal': sine_texture,
        'Gradient': gradient_texture
    }

# Create texture samples
textures = create_texture_samples()

# Analyze each texture
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

lbp_histograms = {}

for i, (name, texture) in enumerate(textures.items()):
    # Normalize texture to 0-255 range
    texture_norm = ((texture - texture.min()) / (texture.max() - texture.min()) * 255).astype(np.uint8)
    
    # Apply LBP
    lbp, hist = analyze_texture_with_lbp(texture_norm)
    lbp_histograms[name] = hist
    
    # Original texture
    axes[0, i].imshow(texture, cmap='gray')
    axes[0, i].set_title(f'{name} Texture')
    axes[0, i].axis('off')
    
    # LBP image
    axes[1, i].imshow(lbp, cmap='gray')
    axes[1, i].set_title(f'{name} LBP')
    axes[1, i].axis('off')
    
    # LBP histogram
    axes[2, i].bar(range(len(hist)), hist, alpha=0.7)
    axes[2, i].set_title(f'{name} LBP Histogram')
    axes[2, i].set_xlabel('LBP Pattern')
    axes[2, i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Compare histograms quantitatively
print("Texture Discrimination using LBP Histograms:")
print("-" * 60)

texture_names = list(textures.keys())
for i in range(len(texture_names)):
    for j in range(i+1, len(texture_names)):
        name1, name2 = texture_names[i], texture_names[j]
        hist1, hist2 = lbp_histograms[name1], lbp_histograms[name2]
        
        # Calculate chi-square distance
        chi2_distance = np.sum((hist1 - hist2)**2 / (hist1 + hist2 + 1e-7))
        
        print(f"{name1:12s} vs {name2:12s}: Chi-square distance = {chi2_distance:.4f}")

---

## 🎯 Part 5: Advanced Feature Descriptors - SIFT and ORB

Let's explore modern feature descriptors that are widely used in computer vision applications.

In [ ]:
def extract_sift_orb_features(image):
    """Extract SIFT and ORB features from an image."""
    
    # Convert to uint8 if needed
    if image.dtype != np.uint8:
        image_uint8 = (image * 255).astype(np.uint8)
    else:
        image_uint8 = image
    
    # SIFT detector
    sift = cv2.SIFT_create()
    sift_keypoints, sift_descriptors = sift.detectAndCompute(image_uint8, None)
    
    # ORB detector
    orb = cv2.ORB_create(nfeatures=500)
    orb_keypoints, orb_descriptors = orb.detectAndCompute(image_uint8, None)
    
    return {
        'sift': {'keypoints': sift_keypoints, 'descriptors': sift_descriptors},
        'orb': {'keypoints': orb_keypoints, 'descriptors': orb_descriptors}
    }

def visualize_keypoints(image, features):
    """Visualize detected keypoints."""
    
    if image.dtype != np.uint8:
        image_uint8 = (image * 255).astype(np.uint8)
    else:
        image_uint8 = image
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Original image
    axes[0].imshow(image, cmap='gray')
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    # SIFT keypoints
    sift_image = cv2.drawKeypoints(image_uint8, features['sift']['keypoints'], None,
                                  flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    axes[1].imshow(cv2.cvtColor(sift_image, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f'SIFT Keypoints ({len(features["sift"]["keypoints"])} detected)')
    axes[1].axis('off')
    
    # ORB keypoints
    orb_image = cv2.drawKeypoints(image_uint8, features['orb']['keypoints'], None,
                                 color=(0, 255, 0))
    axes[2].imshow(cv2.cvtColor(orb_image, cv2.COLOR_BGR2RGB))
    axes[2].set_title(f'ORB Keypoints ({len(features["orb"]["keypoints"])} detected)')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

# Extract features from our test image
features = extract_sift_orb_features(image)
visualize_keypoints(image, features)

# Analyze keypoint properties
print("Feature Detector Comparison:")
print("-" * 50)
print(f"SIFT keypoints detected: {len(features['sift']['keypoints'])}")
print(f"ORB keypoints detected: {len(features['orb']['keypoints'])}")

if features['sift']['descriptors'] is not None:
    print(f"SIFT descriptor size: {features['sift']['descriptors'].shape[1]} dimensions")
if features['orb']['descriptors'] is not None:
    print(f"ORB descriptor size: {features['orb']['descriptors'].shape[1]} dimensions")

# Analyze keypoint scale distribution for SIFT
if len(features['sift']['keypoints']) > 0:
    sift_scales = [kp.size for kp in features['sift']['keypoints']]
    print(f"\nSIFT scale distribution:")
    print(f"  Mean scale: {np.mean(sift_scales):.2f}")
    print(f"  Scale range: [{np.min(sift_scales):.2f}, {np.max(sift_scales):.2f}]")
    
    # Plot scale distribution
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.hist(sift_scales, bins=20, alpha=0.7, color='blue')
    plt.title('SIFT Keypoint Scale Distribution')
    plt.xlabel('Scale')
    plt.ylabel('Count')
    
    # Analyze orientation distribution
    sift_orientations = [kp.angle for kp in features['sift']['keypoints']]
    plt.subplot(1, 2, 2)
    plt.hist(sift_orientations, bins=36, alpha=0.7, color='green')
    plt.title('SIFT Keypoint Orientation Distribution')
    plt.xlabel('Orientation (degrees)')
    plt.ylabel('Count')
    
    plt.tight_layout()
    plt.show()

---

## 🔗 Part 6: Feature Matching and Applications

Now let's see how features can be used for practical applications like image matching and object recognition.

In [ ]:
def create_transformed_image(image, scale=0.8, rotation=30, translation=(50, 30)):
    """Create a transformed version of an image for matching demo."""
    
    # Create transformation matrix
    center = np.array(image.shape) / 2
    tform = transform.AffineTransform(scale=(scale, scale), 
                                     rotation=np.radians(rotation),
                                     translation=translation)
    
    # Apply transformation
    transformed = transform.warp(image, tform.inverse, output_shape=image.shape)
    
    return transformed

def match_features_demo(image1, image2):
    """Demonstrate feature matching between two images."""
    
    # Convert to uint8
    img1_uint8 = (image1 * 255).astype(np.uint8) if image1.dtype != np.uint8 else image1
    img2_uint8 = (image2 * 255).astype(np.uint8) if image2.dtype != np.uint8 else image2
    
    # Extract SIFT features
    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(img1_uint8, None)
    kp2, des2 = sift.detectAndCompute(img2_uint8, None)
    
    if des1 is None or des2 is None:
        print("Could not extract features from one or both images")
        return
    
    # Match features using FLANN
    FLANN_INDEX_KDTREE = 1
    index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)
    
    matches = flann.knnMatch(des1, des2, k=2)
    
    # Apply ratio test
    good_matches = []
    for match_pair in matches:
        if len(match_pair) == 2:
            m, n = match_pair
            if m.distance < 0.7 * n.distance:
                good_matches.append(m)
    
    # Visualize matches
    if len(good_matches) > 4:
        # Draw matches
        match_img = cv2.drawMatches(img1_uint8, kp1, img2_uint8, kp2, 
                                   good_matches[:20], None, 
                                   flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        
        plt.figure(figsize=(15, 8))
        plt.imshow(cv2.cvtColor(match_img, cv2.COLOR_BGR2RGB))
        plt.title(f'Feature Matching: {len(good_matches)} good matches found')
        plt.axis('off')
        plt.show()
        
        return good_matches, kp1, kp2
    else:
        print(f"Not enough good matches found: {len(good_matches)}")
        return None, kp1, kp2

# Create transformed image for matching
transformed_image = create_transformed_image(image)

# Display original and transformed images
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(image, cmap='gray')
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(transformed_image, cmap='gray')
plt.title('Transformed Image\n(Scaled, Rotated, Translated)')
plt.axis('off')

plt.tight_layout()
plt.show()

# Perform feature matching
matches, kp1, kp2 = match_features_demo(image, transformed_image)

if matches is not None:
    print(f"\nMatching Results:")
    print(f"Keypoints in image 1: {len(kp1)}")
    print(f"Keypoints in image 2: {len(kp2)}")
    print(f"Good matches found: {len(matches)}")
    print(f"Matching ratio: {len(matches) / min(len(kp1), len(kp2)):.3f}")

---

## 🎓 Student Activity: Build a Simple Object Detection System

**Time: 15 minutes**

Your task is to create a simple object detection system that can identify specific objects in images using feature matching.

### Instructions:

1. **Choose a template object** from the camera image (e.g., a distinctive region)
2. **Extract features** from the template using SIFT or ORB
3. **Search for the template** in a larger scene image
4. **Visualize the detection results**

### Hints:
- Use `image[y1:y2, x1:x2]` to extract a region of interest
- You can use the functions we've already created
- Try different feature detectors and see which works best
- Think about how to determine if a match is good enough

In [ ]:
# Student Activity Workspace
# Your code here!

# Step 1: Extract a template from the camera image
# Example: template = image[100:200, 150:250]  # Adjust coordinates as needed

# Step 2: Create a scene image (you can use the full camera image or create a composite)

# Step 3: Extract features from both template and scene

# Step 4: Match features and find the template in the scene

# Step 5: Visualize your results

print("Start implementing your object detection system here!")
print("Remember to:")
print("1. Extract a good template region")
print("2. Use feature detection and matching")
print("3. Visualize your results")
print("4. Evaluate the quality of your detection")

### 🔍 Solution (Run this cell to reveal the solution)

In [ ]:
#@title Click to show solution

def simple_object_detection(scene_image, template_region):
    """Simple object detection using feature matching."""
    
    # Extract template
    template = scene_image[template_region[0]:template_region[1], 
                          template_region[2]:template_region[3]]
    
    # Convert to uint8
    scene_uint8 = (scene_image * 255).astype(np.uint8) if scene_image.dtype != np.uint8 else scene_image
    template_uint8 = (template * 255).astype(np.uint8) if template.dtype != np.uint8 else template
    
    # Extract features
    sift = cv2.SIFT_create()
    kp_template, des_template = sift.detectAndCompute(template_uint8, None)
    kp_scene, des_scene = sift.detectAndCompute(scene_uint8, None)
    
    if des_template is None or des_scene is None:
        print("Could not extract features")
        return None
    
    # Match features
    bf = cv2.BFMatcher()
    matches = bf.knnMatch(des_template, des_scene, k=2)
    
    # Apply ratio test
    good_matches = []
    for match_pair in matches:
        if len(match_pair) == 2:
            m, n = match_pair
            if m.distance < 0.75 * n.distance:
                good_matches.append(m)
    
    # Visualize results
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Template
    axes[0].imshow(template, cmap='gray')
    axes[0].set_title(f'Template\n({len(kp_template)} keypoints)')
    axes[0].axis('off')
    
    # Scene
    axes[1].imshow(scene_image, cmap='gray')
    axes[1].set_title(f'Scene\n({len(kp_scene)} keypoints)')
    axes[1].axis('off')
    
    # Mark template region in scene
    rect = plt.Rectangle((template_region[2], template_region[0]), 
                        template_region[3] - template_region[2],
                        template_region[1] - template_region[0],
                        fill=False, color='red', linewidth=2)
    axes[1].add_patch(rect)
    
    # Matches
    if len(good_matches) > 4:
        match_img = cv2.drawMatches(template_uint8, kp_template, 
                                   scene_uint8, kp_scene, 
                                   good_matches[:10], None,
                                   flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        axes[2].imshow(cv2.cvtColor(match_img, cv2.COLOR_BGR2RGB))
        axes[2].set_title(f'Matches\n({len(good_matches)} good matches)')
    else:
        axes[2].text(0.5, 0.5, f'Not enough matches\n({len(good_matches)} found)', 
                    ha='center', va='center', transform=axes[2].transAxes, fontsize=14)
        axes[2].set_title('Detection Failed')
    
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return len(good_matches)

# Example usage
print("Solution: Simple Object Detection System")
print("="*50)

# Define template region (y1, y2, x1, x2) - adjust these coordinates
template_region = (100, 200, 150, 250)  # Example coordinates

# Run detection
num_matches = simple_object_detection(image, template_region)

if num_matches is not None:
    print(f"\nDetection completed with {num_matches} matches")
    if num_matches > 10:
        print("✅ Good detection - object likely found!")
    elif num_matches > 5:
        print("⚠️ Moderate detection - object possibly found")
    else:
        print("❌ Poor detection - object not found or template not distinctive")

print("\nTry adjusting the template_region coordinates to select different parts of the image!")

---

## 🏥 Part 7: Real-World Application - Medical Image Analysis

Let's explore a practical application of feature extraction in medical imaging.

In [ ]:
def create_synthetic_xray():
    """Create a synthetic chest X-ray for demonstration."""
    size = 256
    x = np.linspace(-1, 1, size)
    y = np.linspace(-1, 1, size)
    X, Y = np.meshgrid(x, y)
    
    # Create chest outline
    chest = np.ones((size, size))
    
    # Add lung regions
    left_lung = ((X + 0.3)**2 + Y**2 < 0.25) & (X < 0)
    right_lung = ((X - 0.3)**2 + Y**2 < 0.25) & (X > 0)
    chest[left_lung | right_lung] = 0.7
    
    # Add ribs
    for i in range(5):
        rib_y = -0.6 + i * 0.3
        rib_mask = (np.abs(Y - rib_y) < 0.02) & (np.abs(X) < 0.7)
        chest[rib_mask] = 0.3
    
    # Add heart region
    heart = ((X + 0.1)**2 + (Y + 0.2)**2 < 0.08)
    chest[heart] = 0.4
    
    # Add some abnormalities (nodules)
    nodule1 = ((X + 0.2)**2 + (Y - 0.1)**2 < 0.01)
    nodule2 = ((X - 0.15)**2 + (Y + 0.25)**2 < 0.008)
    chest[nodule1 | nodule2] = 0.2
    
    # Add noise
    noise = np.random.normal(0, 0.05, chest.shape)
    chest = np.clip(chest + noise, 0, 1)
    
    return chest

def analyze_medical_image(image):
    """Analyze medical image using various feature extraction techniques."""
    
    # Convert to uint8
    image_uint8 = (image * 255).astype(np.uint8)
    
    # Edge detection for structure analysis
    edges = feature.canny(image, sigma=1.5, low_threshold=0.1, high_threshold=0.2)
    
    # Blob detection for abnormality detection
    blobs = feature.blob_log(image, min_sigma=2, max_sigma=10, num_sigma=5, threshold=0.1)
    
    # Texture analysis using LBP
    lbp = feature.local_binary_pattern(image_uint8, P=8, R=2, method='uniform')
    
    # Region segmentation (simplified lung segmentation)
    lung_mask = (image > 0.5) & (image < 0.8)
    lung_mask = morphology.binary_opening(lung_mask, morphology.disk(3))
    lung_mask = morphology.binary_closing(lung_mask, morphology.disk(8))
    
    return edges, blobs, lbp, lung_mask

# Create synthetic X-ray
xray_image = create_synthetic_xray()

# Analyze the image
edges, blobs, lbp, lung_mask = analyze_medical_image(xray_image)

# Visualize analysis results
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Original X-ray
axes[0, 0].imshow(xray_image, cmap='gray')
axes[0, 0].set_title('Synthetic Chest X-ray')
axes[0, 0].axis('off')

# Edge detection
axes[0, 1].imshow(edges, cmap='gray')
axes[0, 1].set_title('Anatomical Structure (Edges)')
axes[0, 1].axis('off')

# Blob detection for abnormalities
axes[0, 2].imshow(xray_image, cmap='gray')
for blob in blobs:
    y, x, r = blob
    circle = plt.Circle((x, y), r, color='red', fill=False, linewidth=2)
    axes[0, 2].add_patch(circle)
axes[0, 2].set_title(f'Abnormality Detection ({len(blobs)} found)')
axes[0, 2].axis('off')

# Texture analysis
axes[1, 0].imshow(lbp, cmap='viridis')
axes[1, 0].set_title('Texture Analysis (LBP)')
axes[1, 0].axis('off')

# Lung segmentation
axes[1, 1].imshow(lung_mask, cmap='gray')
axes[1, 1].set_title('Lung Region Segmentation')
axes[1, 1].axis('off')

# Combined analysis
combined = np.zeros((*xray_image.shape, 3))
combined[:, :, 0] = xray_image  # Red channel: original
combined[:, :, 1] = lung_mask.astype(float) * 0.5  # Green channel: lungs
combined[:, :, 2] = edges.astype(float) * 0.3  # Blue channel: edges

axes[1, 2].imshow(combined)
axes[1, 2].set_title('Combined Analysis')
axes[1, 2].axis('off')

# Add annotations for detected abnormalities
for i, blob in enumerate(blobs):
    y, x, r = blob
    circle = plt.Circle((x, y), r, color='yellow', fill=False, linewidth=2)
    axes[1, 2].add_patch(circle)
    axes[1, 2].text(x, y-r-5, f'A{i+1}', color='yellow', fontweight='bold', ha='center')

plt.tight_layout()
plt.show()

# Analysis report
print("Medical Image Analysis Report:")
print("="*50)
print(f"Image size: {xray_image.shape}")
print(f"Detected anatomical structures: {np.sum(edges)} edge pixels")
print(f"Potential abnormalities detected: {len(blobs)}")
print(f"Lung region area: {np.sum(lung_mask)} pixels ({np.sum(lung_mask)/lung_mask.size*100:.1f}% of image)")

if len(blobs) > 0:
    print("\nDetailed abnormality analysis:")
    for i, blob in enumerate(blobs):
        y, x, r = blob
        print(f"  Abnormality {i+1}: Position=({x:.0f}, {y:.0f}), Size={r:.1f}")
        
        # Check if it's in lung region
        if lung_mask[int(y), int(x)]:
            print(f"                   ⚠️  Located in lung tissue - requires attention")
        else:
            print(f"                   ℹ️  Located outside lung tissue")

---

## 📊 Part 8: Performance Analysis and Best Practices

Let's analyze the computational performance of different feature extraction methods and discuss best practices.

In [ ]:
import time

def benchmark_feature_methods(image, num_runs=5):
    """Benchmark different feature extraction methods."""
    
    # Convert to uint8
    image_uint8 = (image * 255).astype(np.uint8) if image.dtype != np.uint8 else image
    
    methods = {
        'Sobel Edge Detection': lambda: filters.sobel(image),
        'Canny Edge Detection': lambda: feature.canny(image, sigma=1.0),
        'Harris Corner Detection': lambda: feature.corner_harris(image),
        'LBP Texture Analysis': lambda: feature.local_binary_pattern(image_uint8, P=8, R=1),
        'SIFT Feature Detection': lambda: cv2.SIFT_create().detectAndCompute(image_uint8, None),
        'ORB Feature Detection': lambda: cv2.ORB_create().detectAndCompute(image_uint8, None)
    }
    
    results = {}
    
    print("Benchmarking Feature Extraction Methods...")
    print("-" * 60)
    
    for method_name, method_func in methods.items():
        times = []
        
        for _ in range(num_runs):
            start_time = time.time()
            try:
                result = method_func()
                end_time = time.time()
                times.append(end_time - start_time)
            except Exception as e:
                print(f"Error in {method_name}: {e}")
                times.append(float('inf'))
        
        avg_time = np.mean(times)
        std_time = np.std(times)
        results[method_name] = {'avg': avg_time, 'std': std_time}
        
        print(f"{method_name:<25}: {avg_time*1000:6.2f} ± {std_time*1000:5.2f} ms")
    
    return results

def visualize_performance(results):
    """Visualize performance benchmarking results."""
    
    methods = list(results.keys())
    avg_times = [results[method]['avg'] * 1000 for method in methods]  # Convert to ms
    std_times = [results[method]['std'] * 1000 for method in methods]
    
    plt.figure(figsize=(12, 6))
    bars = plt.bar(range(len(methods)), avg_times, yerr=std_times, 
                   capsize=5, alpha=0.7, color=plt.cm.viridis(np.linspace(0, 1, len(methods))))
    
    plt.xlabel('Feature Extraction Methods')
    plt.ylabel('Execution Time (ms)')
    plt.title('Performance Comparison of Feature Extraction Methods')
    plt.xticks(range(len(methods)), [m.replace(' ', '\n') for m in methods], rotation=0)
    plt.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, avg_time in zip(bars, avg_times):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                f'{avg_time:.1f}ms', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()

# Run performance benchmark
performance_results = benchmark_feature_methods(image)
visualize_performance(performance_results)

# Memory usage analysis
print("\nMemory Usage Analysis:")
print("-" * 40)
print(f"Original image: {image.nbytes / 1024:.1f} KB")

# Analyze output sizes
sobel_result = filters.sobel(image)
canny_result = feature.canny(image)
lbp_result = feature.local_binary_pattern((image*255).astype(np.uint8), P=8, R=1)

print(f"Sobel edges: {sobel_result.nbytes / 1024:.1f} KB")
print(f"Canny edges: {canny_result.nbytes / 1024:.1f} KB")
print(f"LBP texture: {lbp_result.nbytes / 1024:.1f} KB")

# SIFT/ORB descriptor sizes
sift = cv2.SIFT_create()
orb = cv2.ORB_create()

image_uint8 = (image * 255).astype(np.uint8)
_, sift_desc = sift.detectAndCompute(image_uint8, None)
_, orb_desc = orb.detectAndCompute(image_uint8, None)

if sift_desc is not None:
    print(f"SIFT descriptors: {sift_desc.nbytes / 1024:.1f} KB ({sift_desc.shape[0]} × {sift_desc.shape[1]})")
if orb_desc is not None:
    print(f"ORB descriptors: {orb_desc.nbytes / 1024:.1f} KB ({orb_desc.shape[0]} × {orb_desc.shape[1]})")

### 🎯 Best Practices for Feature Extraction

Based on our analysis, here are key recommendations:

In [ ]:
def demonstrate_best_practices():
    """Demonstrate best practices for feature extraction."""
    
    print("🎯 FEATURE EXTRACTION BEST PRACTICES")
    print("=" * 50)
    
    print("\n1. CHOOSE THE RIGHT METHOD FOR YOUR APPLICATION:")
    print("   • Real-time applications: Use FAST, ORB, or simple edge detection")
    print("   • High accuracy needed: Use SIFT, SURF, or Canny edges")
    print("   • Texture analysis: Use LBP, Gabor filters, or HOG")
    print("   • Geometric shapes: Use Hough transforms")
    
    print("\n2. PREPROCESSING IS CRUCIAL:")
    print("   • Always normalize your images (0-255 or 0-1 range)")
    print("   • Consider noise reduction (Gaussian smoothing)")
    print("   • Ensure proper contrast (histogram equalization if needed)")
    
    print("\n3. PARAMETER TUNING:")
    print("   • Test different parameters on your specific dataset")
    print("   • Use validation data to avoid overfitting")
    print("   • Consider multi-scale approaches for robustness")
    
    print("\n4. PERFORMANCE OPTIMIZATION:")
    print("   • Profile your code to identify bottlenecks")
    print("   • Use appropriate data types (uint8 vs float64)")
    print("   • Consider parallel processing for batch operations")
    
    print("\n5. VALIDATION AND TESTING:")
    print("   • Test on diverse datasets")
    print("   • Measure both accuracy and computational cost")
    print("   • Consider edge cases and failure modes")
    
    # Practical example: Image preprocessing pipeline
    print("\n📋 EXAMPLE: ROBUST PREPROCESSING PIPELINE")
    print("-" * 50)
    
    def robust_preprocessing(image):
        """Example of robust image preprocessing."""
        
        # Step 1: Ensure proper data type
        if image.dtype != np.uint8:
            if image.max() <= 1.0:
                image = (image * 255).astype(np.uint8)
            else:
                image = image.astype(np.uint8)
        
        # Step 2: Noise reduction
        denoised = cv2.GaussianBlur(image, (3, 3), 0.5)
        
        # Step 3: Contrast enhancement (if needed)
        if image.std() < 50:  # Low contrast image
            enhanced = cv2.equalizeHist(denoised)
        else:
            enhanced = denoised
        
        return enhanced
    
    # Demonstrate preprocessing
    original = (image * 255).astype(np.uint8)
    preprocessed = robust_preprocessing(image)
    
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 3, 1)
    plt.imshow(original, cmap='gray')
    plt.title('Original Image')
    plt.axis('off')
    
    plt.subplot(1, 3, 2)
    plt.imshow(preprocessed, cmap='gray')
    plt.title('Preprocessed Image')
    plt.axis('off')
    
    # Show the difference
    difference = np.abs(original.astype(float) - preprocessed.astype(float))
    plt.subplot(1, 3, 3)
    plt.imshow(difference, cmap='hot')
    plt.title('Preprocessing Changes')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Original image stats: Mean={original.mean():.1f}, Std={original.std():.1f}")
    print(f"Preprocessed stats:   Mean={preprocessed.mean():.1f}, Std={preprocessed.std():.1f}")
    print(f"Enhancement applied:  {preprocessed.std() > original.std()}")

demonstrate_best_practices()

---

## 🎉 Summary and Key Takeaways

Congratulations! You've completed the Feature Extraction and Representation workshop. Let's summarize what we've learned:

In [ ]:
def workshop_summary():
    """Provide a comprehensive summary of the workshop."""
    
    print("🎓 FEATURE EXTRACTION WORKSHOP SUMMARY")
    print("=" * 60)
    
    topics_covered = [
        "Edge Detection (Sobel, Prewitt, Roberts, Canny)",
        "Hough Transform for Geometric Shape Detection",
        "Texture Analysis using Local Binary Patterns",
        "Advanced Feature Descriptors (SIFT, ORB)",
        "Feature Matching and Object Detection",
        "Real-world Applications (Medical Imaging)",
        "Performance Analysis and Best Practices"
    ]
    
    print("\n📚 TOPICS COVERED:")
    for i, topic in enumerate(topics_covered, 1):
        print(f"   {i}. {topic}")
    
    key_concepts = [
        "Gradient-based edge detection finds intensity changes",
        "Canny edge detection provides optimal edge localization",
        "Hough transform detects parametric shapes robustly",
        "LBP captures local texture patterns effectively",
        "SIFT features are scale and rotation invariant",
        "Feature matching enables object recognition",
        "Preprocessing significantly affects results",
        "Method selection depends on application requirements"
    ]
    
    print("\n🔑 KEY CONCEPTS:")
    for concept in key_concepts:
        print(f"   • {concept}")
    
    practical_skills = [
        "Implement edge detection algorithms from scratch",
        "Apply and tune Canny edge detection parameters",
        "Use Hough transforms for line and circle detection",
        "Extract and analyze texture features with LBP",
        "Work with SIFT and ORB feature descriptors",
        "Build a simple object detection system",
        "Analyze medical images using feature extraction",
        "Benchmark and optimize feature extraction methods"
    ]
    
    print("\n🛠️ PRACTICAL SKILLS DEVELOPED:")
    for skill in practical_skills:
        print(f"   ✓ {skill}")
    
    next_steps = [
        "Experiment with different feature extraction methods on your own images",
        "Implement a complete object recognition pipeline",
        "Explore deep learning-based feature extraction (CNNs)",
        "Study advanced topics like feature pyramids and multi-scale analysis",
        "Apply these techniques to your specific domain or research area"
    ]
    
    print("\n🚀 RECOMMENDED NEXT STEPS:")
    for step in next_steps:
        print(f"   → {step}")
    
    print("\n📖 ADDITIONAL RESOURCES:")
    resources = [
        "OpenCV Documentation: https://docs.opencv.org/",
        "Scikit-image Documentation: https://scikit-image.org/",
        "'Computer Vision: Algorithms and Applications' by Richard Szeliski",
        "'Digital Image Processing' by Gonzalez and Woods",
        "'Computer Vision: A Modern Approach' by Forsyth and Ponce"
    ]
    
    for resource in resources:
        print(f"   📘 {resource}")
    
    print("\n" + "=" * 60)
    print("Thank you for completing the Feature Extraction workshop!")
    print("Continue practicing and exploring these powerful techniques.")
    print("=" * 60)

workshop_summary()

---

## 🏆 Final Challenge (Optional)

**Create a Multi-Feature Analysis System**

Combine multiple feature extraction techniques to create a comprehensive image analysis system. Your system should:

1. **Extract multiple types of features** (edges, corners, texture, keypoints)
2. **Provide quantitative analysis** of each feature type
3. **Create a visual dashboard** showing all analyses
4. **Make recommendations** about which features are most useful for the given image

Use the workspace below to implement your solution!

In [ ]:
# Final Challenge Workspace
# Create your multi-feature analysis system here!

def multi_feature_analysis_system(image):
    """Your comprehensive feature analysis system."""
    
    # TODO: Implement your system here
    # Suggestions:
    # - Extract edges, corners, texture, and keypoints
    # - Quantify each feature type
    # - Create visualizations
    # - Provide analysis recommendations
    
    pass

# Test your system
# multi_feature_analysis_system(image)

print("🏆 Final Challenge: Multi-Feature Analysis System")
print("Implement a comprehensive system that analyzes multiple feature types!")
print("Good luck and have fun exploring!")